# LLM + SAE end-to-end запуск

Ноутбук демонстрирует полный практический пайплайн: загрузка LLM, SAE и обучающего датасета, извлечение активаций `h`, получение латентов `z` и реконструкций `h_hat`, статистические проверки корректности применения SAE и графики для вывода о корректности применения SAE.


In [ ]:
from pathlib import Path
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from scipy import stats
os.environ.setdefault("MPLCONFIGDIR", "/tmp/loupe_matplotlib")
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from interpretability.sae.sae import SAE
from interpretability.sae.feature_analysis import (
    attribute_features_to_concepts,
    feature_indices_for_concept,
    format_feature_token_summary,
    top_tokens_for_feature,
)
from utils.inference_utils.llm import LLM
from utils.stat_utils import (
    cosine_per_valid_token,
    distribution_preservation_statistics,
    flatten_valid_tokens,
    kl_divergence,
    next_token_distribution,
    reconstruction_statistics,
    separability_statistics,
    sparsity_statistics,
    target_token_id,
)

sns.set_theme(style="whitegrid")
torch.set_grad_enabled(False)
warnings.filterwarnings("default")


## 1. Конфигурация

Укажите путь к checkpoint SAE, датасету, модели и анализируемому слою. Для Qwen обычно используются имена вида `model.layers.27`.


In [ ]:
MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-3B-Instruct"
LAYER_NAME = "model.layers.27"

DATASET_CSV_PATH = PROJECT_ROOT / "data" / "sae_activation_statistics_train_dataset.csv"
SAE_CHECKPOINT_PATH = PROJECT_ROOT / "models" / "qwen2_5_3b_layer27_sae.pt"

TEXT_COLUMN = "text"
CONCEPT_COLUMN = "concept_label"
TARGET_TOKEN_COLUMN = "target_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
BATCH_SIZE = 2
MAX_SAMPLES = 128
MAX_LENGTH = 256

ALPHA = 0.05
RECONSTRUCTION_NMSE_TAU = 0.20
MMD_EPSILON = 0.05
SPARSITY_THRESHOLD = 1e-8

RUN_CAUSAL_INTERVENTIONS = False
INTERVENTION_VALUE = 2.0
N_INTERVENTION_PROMPTS = 8
N_FEATURES_PER_CONCEPT = 5

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 2. Загрузка датасета и SAE

Датасет должен содержать тексты, на которых обучался SAE, и метки концептов. Минимальные колонки: `text`, `concept_label`. Колонка `target_token` нужна для интервенционного теста.


In [ ]:
def infer_sae_dimensions(state_dict: dict[str, torch.Tensor]) -> tuple[int, int]:
    encoder_weight_keys = [
        "encoder.0.weight",
        "sae.0.0.weight",
        "sae.0.weight",
    ]
    for key in encoder_weight_keys:
        if key in state_dict and state_dict[key].ndim == 2:
            latent_size, hidden_size = state_dict[key].shape
            return int(hidden_size), int(latent_size)
    for key, value in state_dict.items():
        if key.endswith("weight") and value.ndim == 2:
            latent_size, hidden_size = value.shape
            if latent_size > hidden_size:
                return int(hidden_size), int(latent_size)
    raise ValueError("Could not infer SAE dimensions from checkpoint")


def normalize_sae_state_dict(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    normalized = dict(state_dict)
    flat_legacy_mapping = {
        "sae.0.weight": "encoder.0.weight",
        "sae.0.bias": "encoder.0.bias",
        "sae.1.weight": "encoder.1.weight",
        "sae.1.bias": "encoder.1.bias",
        "sae.1.running_mean": "encoder.1.running_mean",
        "sae.1.running_var": "encoder.1.running_var",
        "sae.1.num_batches_tracked": "encoder.1.num_batches_tracked",
        "sae.3.weight": "decoder.weight",
        "sae.3.bias": "decoder.bias",
    }
    nested_legacy_mapping = {
        "sae.0.0.weight": "encoder.0.weight",
        "sae.0.0.bias": "encoder.0.bias",
        "sae.0.1.weight": "encoder.1.weight",
        "sae.0.1.bias": "encoder.1.bias",
        "sae.0.1.running_mean": "encoder.1.running_mean",
        "sae.0.1.running_var": "encoder.1.running_var",
        "sae.0.1.num_batches_tracked": "encoder.1.num_batches_tracked",
        "sae.1.weight": "decoder.weight",
        "sae.1.bias": "decoder.bias",
    }
    legacy_mapping = flat_legacy_mapping if "sae.3.weight" in state_dict else nested_legacy_mapping
    for old_key, new_key in legacy_mapping.items():
        if old_key in state_dict and new_key not in normalized:
            normalized[new_key] = state_dict[old_key]
    return normalized


def load_sae(checkpoint_path: Path, device: str, dtype: torch.dtype) -> SAE:
    state_dict = normalize_sae_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    hidden_size, latent_size = infer_sae_dimensions(state_dict)
    sae = SAE(
        in_hidden_state_size=hidden_size,
        sparse_hidden_state_size=latent_size,
        device=device,
        dtype=dtype,
    )
    missing, unexpected = sae.load_state_dict(state_dict, strict=False)
    if missing:
        print("Missing SAE keys:", missing[:10])
    if unexpected:
        print("Unexpected SAE keys:", unexpected[:10])
    sae.eval()
    return sae


assert DATASET_CSV_PATH.exists(), f"Dataset not found: {DATASET_CSV_PATH}"
assert SAE_CHECKPOINT_PATH.exists(), f"SAE checkpoint not found: {SAE_CHECKPOINT_PATH}"

dataset = pd.read_csv(DATASET_CSV_PATH)
required_columns = {TEXT_COLUMN, CONCEPT_COLUMN}
missing_columns = required_columns - set(dataset.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

dataset = dataset.dropna(subset=[TEXT_COLUMN, CONCEPT_COLUMN]).reset_index(drop=True)
if MAX_SAMPLES is not None:
    dataset = dataset.head(MAX_SAMPLES).copy()

texts = dataset[TEXT_COLUMN].astype(str).tolist()
concept_labels = dataset[CONCEPT_COLUMN].astype(str).tolist()

sae = load_sae(SAE_CHECKPOINT_PATH, DEVICE, DTYPE)
print(f"Loaded {len(dataset)} samples")
print(f"SAE hidden={sae.in_hidden_state_size}, latent={sae.sparse_hidden_state_size}")
dataset.head()

## 3. Инференс LLM + SAE на тренировочном датасете

Из выбранного слоя LLM извлекаются токеновые активации `h`. Затем SAE возвращает `z` и `h_hat`. Для статистики используются только непаддинговые токены.


In [ ]:
llm = LLM(
    model_name_or_path=MODEL_NAME_OR_PATH, 
    device=DEVICE
)
named_modules = dict(llm.model.named_modules())
assert LAYER_NAME in named_modules, f"Layer not found: {LAYER_NAME}"
layer_module = named_modules[LAYER_NAME]


def module_output_to_hidden_state(output):
    if torch.is_tensor(output):
        return output
    if isinstance(output, (tuple, list)):
        return output[0]
    raise TypeError(f"Unsupported layer output type: {type(output)}")


def extract_layer_activations(llm: LLM, texts: list[str], layer_name: str, batch_size: int):
    module = dict(llm.model.named_modules())[layer_name]
    all_hidden_states = []
    all_attention_masks = []
    all_tokens = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = llm.tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
        ).to(llm.device)
        saved_hidden_states = []

        def hook_fn(module, inputs, output):
            saved_hidden_states.append(module_output_to_hidden_state(output).detach().cpu())

        handle = module.register_forward_hook(hook_fn)
        try:
            with torch.no_grad():
                _ = llm.model(**inputs)
        finally:
            handle.remove()

        hidden_states = saved_hidden_states[-1]
        attention_mask = inputs["attention_mask"].detach().cpu().bool()
        tokens = [llm.tokenizer.convert_ids_to_tokens(row) for row in inputs["input_ids"].detach().cpu()]

        all_hidden_states.append(hidden_states)
        all_attention_masks.append(attention_mask)
        all_tokens.extend(tokens)

    return torch.cat(all_hidden_states, dim=0), torch.cat(all_attention_masks, dim=0), all_tokens


h, attention_mask, tokens = extract_layer_activations(llm, texts, LAYER_NAME, BATCH_SIZE)
with torch.no_grad():
    sae_output = sae(h.to(DEVICE), return_output=True)

z = sae_output.latent_activation.detach().cpu()
h_hat = sae_output.reconstructed_hidden_state.detach().cpu()

# Padding tokens should not dominate token-level feature discovery.
z = z.masked_fill(~attention_mask.unsqueeze(-1), 0.0)
h_hat = h_hat.masked_fill(~attention_mask.unsqueeze(-1), 0.0)

print("h:", tuple(h.shape), "z:", tuple(z.shape), "h_hat:", tuple(h_hat.shape))

## 4. Проверка 1: сохранение информации

Гипотеза: `NMSE_SAE < tau_R`, а косинусная близость `h` и `h_hat` должна быть высокой.


In [ ]:
reconstruction_results, reconstruction_error_values = reconstruction_statistics(
    sae=sae,
    original=h,
    reconstructed=h_hat,
    mask=attention_mask,
    nmse_tau=RECONSTRUCTION_NMSE_TAU,
    alpha=ALPHA,
)
cosine_values = cosine_per_valid_token(h, h_hat, attention_mask)
reconstruction_results


## 5. Проверка 2: сохранение распределения активаций

Сравниваются распределения исходных `h` и восстановленных `h_hat`: MMD, KS по координатам, Wasserstein distance и JSD по гистограммам.


In [ ]:
valid_h = flatten_valid_tokens(h, attention_mask)
valid_h_hat = flatten_valid_tokens(h_hat, attention_mask)
valid_z = flatten_valid_tokens(z, attention_mask)

distribution_results, distribution_details = distribution_preservation_statistics(
    original=h,
    reconstructed=h_hat,
    mask=attention_mask,
    mmd_epsilon=MMD_EPSILON,
    alpha=ALPHA,
)

ks_statistics = distribution_details["ks_statistics"]
wasserstein_values = distribution_details["wasserstein_values"]
mmd_value = distribution_results["mmd"]
jsd_value = distribution_results["histogram_jsd"]

distribution_results


## 6. Проверка 3: разреженность латентного пространства

Гипотеза: `z` должен быть более разреженным, чем исходное плотное представление `h`. Сравниваем L0/share активных фичей, Hoyer sparsity и normalized entropy.


In [ ]:
sparsity_results, sparsity_details = sparsity_statistics(
    sae=sae,
    original=h,
    latent=z,
    mask=attention_mask,
    threshold=SPARSITY_THRESHOLD,
    alpha=ALPHA,
)

z_hoyer = sparsity_details["z_hoyer"]
h_hoyer = sparsity_details["h_hoyer"]

sparsity_results


## 7. Проверка 4: разделимость концептов в `z`

Для концептов используются метки `concept_label`. Сравниваем sample-level представления `h` и `z` через silhouette score и строим таблицу SAE-фичей-кандидатов для каждого концепта.


In [ ]:
separability_results, sample_h_tensor, sample_z_tensor = separability_statistics(
    original=h,
    latent=z,
    mask=attention_mask,
    concept_labels=concept_labels,
)

sample_h = sample_h_tensor.numpy()
sample_z = sample_z_tensor.numpy()
unique_concepts = sorted(set(concept_labels))
silhouette_h = separability_results["silhouette_h"]
silhouette_z = separability_results["silhouette_z"]

feature_attributions = attribute_features_to_concepts(
    latent_activations=sample_z_tensor,
    concept_labels=concept_labels,
    top_k=10,
    score_method="cohen_d",
)
feature_attributions_df = pd.DataFrame([item.to_dict() for item in feature_attributions])

display(separability_results)
feature_attributions_df.head(20)


## 8. Token-based атрибуция фичей

Этот блок показывает, какие конкретные токены сильнее всего активировали выбранные фичи. Summary можно передать более крупной LLM для объединения токенов и контекстов в семантическую категорию.


In [ ]:
selected_concept = unique_concepts[0]
selected_features = feature_indices_for_concept(
    latent_activations=sample_z_tensor,
    concept_labels=concept_labels,
    concept_label=selected_concept,
    top_k=3,
    score_method="cohen_d",
)

token_attributions = top_tokens_for_feature(
    latent_activations=z,
    concept_labels=concept_labels,
    feature_index=selected_features[0],
    tokens=tokens,
    top_k=30,
    concept_label=selected_concept,
    context_window=3,
)

print(format_feature_token_summary(token_attributions, feature_index=selected_features[0], max_rows=15))


## 9. Проверка 5: каузальная селективность интервенций

Этот тест дорогой, поэтому по умолчанию отключен. Он сравнивает `Q_SAE` для фичей-кандидатов с baseline из случайных SAE-фичей: `Q = delta_target / (delta_side + eps)`, где `delta_side` — KL-дивергенция распределений следующего токена.


In [ ]:
causal_results = []
if RUN_CAUSAL_INTERVENTIONS:
    assert TARGET_TOKEN_COLUMN in dataset.columns, f"{TARGET_TOKEN_COLUMN} is required for causal interventions"
    llm.clear_sae_hooks()
    latent_size = sae.sparse_hidden_state_size

    intervention_rows = dataset.dropna(subset=[TARGET_TOKEN_COLUMN]).head(N_INTERVENTION_PROMPTS)
    for row in intervention_rows.itertuples(index=False):
        prompt = str(getattr(row, TEXT_COLUMN))
        concept = str(getattr(row, CONCEPT_COLUMN))
        target = str(getattr(row, TARGET_TOKEN_COLUMN))
        target_id = target_token_id(llm, target)

        concept_features = feature_indices_for_concept(
            latent_activations=sample_z_tensor,
            concept_labels=concept_labels,
            concept_label=concept,
            top_k=N_FEATURES_PER_CONCEPT,
            score_method="cohen_d",
        )
        random_features = random.sample(range(latent_size), k=len(concept_features))

        p_base = next_token_distribution(llm, prompt, max_length=MAX_LENGTH)

        layer_handle = llm.add_sae(
            sae=sae,
            layer_name=LAYER_NAME,
            feature_indices=concept_features,
            intervention_value=INTERVENTION_VALUE,
            mode="add",
            token_positions=[-1],
        )
        p_sae = next_token_distribution(llm, prompt, max_length=MAX_LENGTH)
        llm.remove_sae(layer_handle)

        layer_handle = llm.add_sae(
            sae=sae,
            layer_name=LAYER_NAME,
            feature_indices=random_features,
            intervention_value=INTERVENTION_VALUE,
            mode="add",
            token_positions=[-1],
        )
        p_random = next_token_distribution(llm, prompt, max_length=MAX_LENGTH)
        llm.remove_sae(layer_handle)

        delta_target_sae = float((p_sae[target_id] - p_base[target_id]).item())
        delta_target_random = float((p_random[target_id] - p_base[target_id]).item())
        side_sae = kl_divergence(p_base, p_sae)
        side_random = kl_divergence(p_base, p_random)

        causal_results.append({
            "concept_label": concept,
            "target_token": target,
            "q_sae": delta_target_sae / (side_sae + 1e-8),
            "q_random": delta_target_random / (side_random + 1e-8),
            "delta_target_sae": delta_target_sae,
            "delta_target_random": delta_target_random,
            "side_sae": side_sae,
            "side_random": side_random,
        })

causal_results_df = pd.DataFrame(causal_results)
if len(causal_results_df) > 1:
    q_test = stats.wilcoxon(causal_results_df["q_sae"] - causal_results_df["q_random"], alternative="greater")
    causal_summary = {
        "mean_q_sae": float(causal_results_df["q_sae"].mean()),
        "mean_q_random": float(causal_results_df["q_random"].mean()),
        "wilcoxon_pvalue": float(q_test.pvalue),
        "passed": bool(q_test.pvalue < ALPHA and causal_results_df["q_sae"].mean() > causal_results_df["q_random"].mean()),
    }
else:
    causal_summary = {"passed": None, "reason": "RUN_CAUSAL_INTERVENTIONS=False or not enough prompts"}

display(causal_summary)
causal_results_df.head()


## 10. Графики


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.histplot(reconstruction_error_values, bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Reconstruction MSE по токенам")

sns.histplot(cosine_values, bins=30, ax=axes[0, 1])
axes[0, 1].set_title("Cosine similarity h vs h_hat")

distribution_plot = pd.DataFrame({
    "metric": ["MMD", "mean KS", "mean Wasserstein", "JSD"],
    "value": [mmd_value, np.mean(ks_statistics), np.mean(wasserstein_values), jsd_value],
})
sns.barplot(distribution_plot, x="metric", y="value", ax=axes[0, 2])
axes[0, 2].set_title("Близость распределений h и h_hat")

sparsity_plot = pd.DataFrame({
    "space": ["h", "z"],
    "hoyer": [np.mean(h_hoyer), np.mean(z_hoyer)],
})
sns.barplot(sparsity_plot, x="space", y="hoyer", ax=axes[1, 0])
axes[1, 0].set_title("Hoyer sparsity")

separability_plot = pd.DataFrame({
    "space": ["h", "z"],
    "silhouette": [silhouette_h, silhouette_z],
})
sns.barplot(separability_plot, x="space", y="silhouette", ax=axes[1, 1])
axes[1, 1].set_title("Разделимость концептов")

top_attr = feature_attributions_df.groupby("concept_label").head(5)
sns.scatterplot(top_attr, x="feature_index", y="score", hue="concept_label", ax=axes[1, 2])
axes[1, 2].set_title("Top SAE features by concept")

plt.tight_layout()
plt.show()

if len(causal_results_df) > 0:
    causal_long = causal_results_df[["q_sae", "q_random"]].melt(var_name="intervention", value_name="Q")
    plt.figure(figsize=(6, 4))
    sns.boxplot(causal_long, x="intervention", y="Q")
    plt.title("Causal selectivity Q")
    plt.show()


## 11. Итоговый вывод о корректности применения SAE


In [ ]:
summary = pd.DataFrame([
    {"hypothesis": "information_preservation", **reconstruction_results},
    {"hypothesis": "distribution_preservation", **distribution_results},
    {"hypothesis": "latent_sparsity", **sparsity_results},
    {"hypothesis": "concept_separability", **separability_results},
    {"hypothesis": "causal_selectivity", **causal_summary},
])

display(summary)

required_passes = summary[summary["hypothesis"] != "causal_selectivity"]["passed"].dropna().astype(bool)
causal_pass = causal_summary.get("passed")

if required_passes.all() and causal_pass is True:
    conclusion = "SAE passed all statistical checks, including causal selectivity."
elif required_passes.all() and causal_pass is None:
    conclusion = "SAE passed non-causal checks. Causal selectivity is not evaluated in this run."
elif required_passes.all() and causal_pass is False:
    conclusion = "SAE passed reconstruction/distribution/sparsity/separability checks, but causal selectivity was not confirmed."
else:
    conclusion = "SAE did not pass all required non-causal checks. Do not use its z-space for dissertation claims without improving training/configuration."

print(conclusion)